This it the collab notebook to implement a RAG.

The inputs or source/knowledge base are the pdfs in the folder **/raw_documents**.

The pdfs can be text or images.

The vector db is Chroma db under **/dbs** for the hackathon content. For the upload where users can upload their own dcuments, the dbs are maintined per session and under **user_dbs_<session_ids>**. User documents are maintained under **user_docs_<session_id>**.

The hosting is using gradio.

The RAG pipeline is separated into a data ingestion, that populates the db.

The embeddings that creates the embeddings.

The retrieval that retrieves the data from db.

The  generation that generates the output.

# Create requirements. txt file


In [5]:
packages = [#'numpy',
            'pypdf',
            'fastapi',
            'uvicorn',
            'gradio',
            'groq',
            'langchain',
            'langchain-core',
            'langchain_groq',
            'langchain-community',
            'langchain-huggingface',
            'sentence_transformers',
            'chromadb',
            'python-dotenv'
            #'uuid'
            ]

with open("requirements.txt", "w") as f:
  for package in packages:
    f.write(f"{package}\n")

In [6]:
!pip install --upgrade -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.

## Load the Enviornment files that have the keys



In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

from google.colab import userdata
#
groq_key = userdata.get("GROQ_KEY")
hugging_face_key = userdata.get("HUGGING_FACE_KEY")

if groq_key and groq_key.startswith("gsk_"):
  print(f"Groq Key loaded!")
else:
  print(f"Failed to load Groq Key!")

if hugging_face_key and hugging_face_key.startswith("hf_"):
  print(f"Hugging Key loaded!")
else:
  print(f"Failed to load Hugging Face Key!")

Groq Key loaded!
Hugging Key loaded!


## The Data ingestion

In [2]:
from langchain_community.document_loaders import PyPDFLoader
import os
def load_documents(knowledge_base_path:str):
  documents = []
  for file in os.listdir(knowledge_base_path):
    file_path = os.path.join(knowledge_base_path, file)
    loader = PyPDFLoader(file_path)
    doc = loader.load()
    documents.extend(doc)
    print(f"Loaded {len(doc)} pages from {file_path}")

  print(f"Loaded total {len(documents)} pages")
  return documents

## Create Chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunkizer(documents:list, chunk_size:int, chunk_overlap:int):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  chunks = text_splitter.split_documents(documents)
  print(f"Chunizer: num documents: {len(documents)}, Chunk Sz: {chunk_size} Chunk OverLap: {chunk_overlap}")
  print(f"Chunkizer created {len(chunks)} chunks")
  return chunks

## Storing the emeddings in vector DB

In [4]:
import shutil
import os
from langchain_community.vectorstores import Chroma
import chromadb # Added this import

def add_to_vector_db(vector_db_path:str, chunks:list, embedding_model, purge: bool):
  # Create a persistent client for the specific path. This ensures each vector store
  # operates with its own isolated and correctly configured client.
  print(f"Chroma DB used is at  :{vector_db_path}")

  if purge:
    if os.path.exists(vector_db_path):
      print(f"Purging contents of {vector_db_path}")
      shutil.rmtree(vector_db_path) # Remove the directory and its contents
      print(f"Purged contents of {vector_db_path}")

    print(f"Creating new vector store at {vector_db_path} after purge.")
    os.makedirs(vector_db_path, exist_ok=True)
    os.chmod(vector_db_path, 0o755)

    # Let Chroma.from_documents handle the client creation and database initialization
    vector_store = Chroma.from_documents(
        documents = chunks,
        embedding = embedding_model,
        # No explicit client passed here; Chroma.from_documents will create it
        persist_directory = vector_db_path
    )
    print(f"Added {len(chunks)} chunks to new vector store.")
    return vector_store

  # If not purging and directory doesn't exist
  if not os.path.exists(vector_db_path):
    print(f"Creating new vector store at {vector_db_path}.")
    os.makedirs(vector_db_path, exist_ok=True) # Ensure directory exists for persistence
    os.chmod(vector_db_path, 0o755) # Set permissions
    vector_store = Chroma.from_documents(
        documents = chunks,
        embedding = embedding_model,
        # No explicit client here for creation
        persist_directory = vector_db_path
    )
    print(f"Added {len(chunks)} chunks to new vector store.")
  else:
    print(f"Loading existing vector store from {vector_db_path}.")
    # For loading, we explicitly create the client
    client = chromadb.PersistentClient(path=vector_db_path)
    vector_store = Chroma(
        client = client, # Pass the explicit client
        embedding_function=embedding_model,
        persist_directory=vector_db_path
    )

    # Get unique source file paths from the new chunks being added
    new_chunk_sources = set(doc.metadata['source'] for doc in chunks if 'source' in doc.metadata)

    if new_chunk_sources:
        print(f"Found new chunk sources to update: {new_chunk_sources}")
        for source_file in new_chunk_sources:
            # Delete existing documents from this source file in the vector store
            print(f"Deleting existing chunks in DB from source: {source_file}")
            vector_store.delete(where={"source": source_file})
        print(f"Deleted existing chunks for the specified sources.")

    print(f"Adding {len(chunks)} new chunks to existing vector store.")
    vector_store.add_documents(documents=chunks)

  return vector_store

## This for testing individual functions

In [5]:

from langchain_huggingface import HuggingFaceEmbeddings
def load_chunk_add_to_vector(raw_file_path: str, vector_db_path: str):
  # Get the docs
  documents = load_documents(raw_file_path)
  # print(type(documents))

  #chunk the docs
  chunks  = chunkizer(documents= documents, chunk_size= 1000, chunk_overlap=200)
  print(type(chunks))

  # create embedding model
  embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
  print(type(embedding_model))

  # add chunks to vector db
  add_to_vector_db(vector_db_path, chunks=chunks, embedding_model=embedding_model, purge=False)


In [ ]:
load_chunk_add_to_vector(r'/content/raw_documents', r'/content/dbs')
load_chunk_add_to_vector(r'/content/user_docs', r'/content/user_dbs')

## Method to do similarity search

In [6]:
def vector_db_similarity_search(vector_db, query, k=5):
  # results = vector_db.similarity_search(query=query, k=k)
  # Increase diversity, so use lambda_mult=0.1
  results = vector_db.max_marginal_relevance_search(query=query, k=k, lambda_mult=0.1)
  print(f"Similiarity search returned {len(results)} chunks")
  return results

## Use LLM to generate response

In [7]:
from groq import Groq
def generate_response(system_role: str, query: str):
  groq_client = Groq(api_key = groq_key)
  try:
    result = groq_client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = [
            {
                "role":"system",
                "content": system_role
            },
            {
                "role":"user",
                "content":query
            }
        ]
    )
    return result.choices[0].message.content
  except Exception as e:
    print(f"Groq Connection failed: {e}")

## Rag Pipeline usin Langchain Expression Language. Rami was clear, it you don't use the LCEL, it is downhill from there :)

## LangChain Expression Language (LCEL) for RAG Pipeline

LangChain Expression Language (LCEL) offers a flexible and composable way to build complex LLM applications. It allows you to chain together various components (like document loaders, text splitters, embedding models, vector stores, retrievers, and LLMs) into a single, executable pipeline.

**Benefits of using LCEL:**

*   **Modularity:** Each step is a distinct component, making the pipeline easier to understand, test, and maintain.
*   **Readability:** The `|` (pipe) operator makes the flow of data explicit and intuitive.
*   **Flexibility:** Easily swap out components (e.g., try a different embedding model or LLM).
*   **Streaming & Async Support:** LCEL chains inherently support streaming and asynchronous execution, which is crucial for responsive applications.
*   **Tracing & Debugging:** Integrates well with LangSmith for better observability.

Let's refactor your existing RAG components into an LCEL-compatible chain.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# Assuming `embedding_model` is already initialized globally
# e.g., embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

def create_retriever_lcel(vector_db_path: str, embedding_model):
    """Creates a retriever from a persistent ChromaDB vector store."""
    if not os.path.exists(vector_db_path):
        raise FileNotFoundError(f"Vector DB not found at {vector_db_path}")

    client = chromadb.PersistentClient(path=vector_db_path)
    vector_store = Chroma(
        client=client,
        embedding_function=embedding_model,
        persist_directory=vector_db_path
    )
    # Using .as_retriever() directly for simplicity within LCEL
    # We can customize search_kwargs if needed, e.g., search_kwargs={"k": 5, "lambda_mult": 0.1}
    return vector_store.as_retriever(
                        search_type="mmr",
                        search_kwargs={"k": 5})


def format_docs_lcel(docs):
    """Formats a list of documents into a single string for the LLM context."""
    return "\n\n".join([doc.page_content for doc in docs])

# Define the LLM for generation
# groq_key is available from .env loading
# Temperature is 0, add diversity
llm_lcel = ChatGroq(temperature=0, model_name="llama-3.1-8b-instant", groq_api_key=groq_key)

# Define the prompt template for the RAG chain
rag_prompt_lcel = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the following context to answer the user's question. If the answer is not in the context, say 'I cannot find the answer in the provided documents.'\nContext: {context}"),
    ("user", "{question}"),
])

# LCEL RAG Chain construction example
def create_lcel_rag_chain(retriever):
    lcel_rag_chain = (
        {"context": retriever | RunnableLambda(format_docs_lcel), "question": RunnablePassthrough()}
        | rag_prompt_lcel
        | llm_lcel
        | StrOutputParser()
    )
    return lcel_rag_chain

print("LCEL RAG components defined. You can now use `create_retriever_lcel` and `create_lcel_rag_chain` in your `rag_pipeline_lcel` function.")

### Integrating LCEL into a new `rag_pipeline_lcel` function

create a new `rag_pipeline_lcel` function to use this LCEL-based approach, especially for the retrieval and generation steps. This will make the pipeline more concise and leverage LangChain's capabilities more fully.

In [ ]:
# New `rag_pipeline_lcel` function to use LCEL
def rag_pipeline_lcel(document_choice: str, uploaded_files, question: str, session_id: str):
    global embedding_model # Use the globally initialized embedding model
    current_retriever = None

    # Define session-specific paths
    SESSION_USER_DOCS_PATH = os.path.join('./user_docs', session_id)
    SESSION_USER_DBS_PATH = os.path.join('./user_dbs', session_id)

    # Create directories if they don't exist for the current session
    os.makedirs(SESSION_USER_DOCS_PATH, exist_ok=True)
    os.makedirs(SESSION_USER_DBS_PATH, exist_ok=True)

    if document_choice == 'default':
        print(f"[{session_id}] Using default documents and vector DB.")
        if not os.path.exists(DEFAULT_DB_PATH):
          print("Default vector database not found. Creating....")
          load_chunk_add_to_vector(r'./raw_documents', r'./dbs')

        current_retriever = create_retriever_lcel(DEFAULT_DB_PATH, embedding_model)

    elif document_choice == 'upload':
        print(f"[{session_id}] Processing uploaded documents.")
        if not uploaded_files:
            return "Please upload PDF files for processing."
        print(f"{SESSION_USER_DOCS_PATH}\n{SESSION_USER_DBS_PATH}")
        # Clear previous user documents for this session
        print(f"For session {session_id} chroma db path: {SESSION_USER_DOCS_PATH}")
        if os.path.exists(SESSION_USER_DOCS_PATH):
            shutil.rmtree(SESSION_USER_DOCS_PATH)
        os.makedirs(SESSION_USER_DOCS_PATH, exist_ok=True)

        # Save uploaded files to the session's document folder
        saved_file_paths = []
        for uploaded_file in uploaded_files:
            file_name = os.path.basename(uploaded_file.name)
            save_path = os.path.join(SESSION_USER_DOCS_PATH, file_name)
            shutil.copyfile(uploaded_file.name, save_path)
            saved_file_paths.append(save_path)
            print(f"[{session_id}] Saved {file_name} to {SESSION_USER_DOCS_PATH}")

        # Load, chunk, and add to the session's vector DB
        user_documents = load_documents(SESSION_USER_DOCS_PATH)
        user_chunks = chunkizer(documents=user_documents, chunk_size=1000, chunk_overlap=200)
        # Note: add_to_vector_db will return a vector_store, but we need a retriever for LCEL
        vector_store = add_to_vector_db(SESSION_USER_DBS_PATH,
                                             chunks=user_chunks,
                                             embedding_model=embedding_model,
                                             purge=False)
        current_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

    else:
        return "Invalid document choice. Please select 'default' or 'upload'."

    if not current_retriever:
        return "Error: Could not initialize retriever."

    # Create the LCEL RAG chain using the current_retriever
    lcel_rag_chain = create_lcel_rag_chain(current_retriever)

    print(f"[{session_id}] Running LCEL RAG chain for question: {question}")
    response = lcel_rag_chain.invoke(question)

    return response

print("New `rag_pipeline_lcel` function created using LCEL.")

In [ ]:
def  with_lecl():
  import uuid # Added import for uuid
  with gr.Blocks() as demo_lcel:
      gr.Markdown("# LCEL RAG Document Query Interface")

      session_id = gr.State() # To store the session ID

      with gr.Row():
          document_choice = gr.Radio(
              ['default', 'upload'],
              label="Choose Document Source",
              value='default'
          )

      with gr.Row() as upload_section:
          uploaded_files = gr.File(
              label="Upload PDF Documents (for 'upload' option only)",
              file_count="multiple",
              file_types=[".pdf"],
              visible=False # Initially hidden
          )
          upload_btn = gr.Button("Process Uploaded Documents", visible=False)

      def toggle_upload_visibility(choice):
          if choice == 'upload':
              return gr.update(visible=True), gr.update(visible=True)
          else:
              return gr.update(visible=False), gr.update(visible=False)

      document_choice.change(
          toggle_upload_visibility,
          inputs=document_choice,
          outputs=[uploaded_files, upload_btn]
      )

      with gr.Row() as delete_other_user_data:
          clear_user_data_btn = gr.Button("Clear all user data", visible=True)


      def delete_user_data():
        folder_paths = ['./user_docs','./user_dbs']
        deleted_folders = []
        for folder in folder_paths:
            if os.path.exists(folder):
                try:
                    shutil.rmtree(folder)
                    deleted_folders.append(folder)
                    os.makedirs(folder, exist_ok=True) # Recreate empty directory
                except Exception as e:
                    print(f"Error deleting folder {folder}: {e}")
        if deleted_folders:
            return f"Deleted data in: {', '.join(deleted_folders)}"
        else:
            return "No user data folders found to clear."



      with gr.Row():
          question_input = gr.Textbox(
              label="Find me in the documents:",
              placeholder="Enter your question here..."
          )
      with gr.Row():
          submit_btn = gr.Button("Get Answer")
      with gr.Row():
          output_text = gr.Textbox(label="Response", interactive=False, lines=10)

      def process_and_query_lcel(doc_choice, files, query_text, current_session_id):
          if doc_choice == 'upload' and not files:
              return "Please upload documents or select 'default' source."
          print(f"Starting the LCEL RAG for user specified documents...")
          return rag_pipeline_lcel(doc_choice, files, query_text, current_session_id)

      submit_btn.click(
          process_and_query_lcel,
          inputs=[document_choice, uploaded_files, question_input, session_id],
          outputs=output_text
      )

      # Trigger processing for uploaded files if the upload button is clicked
      upload_btn.click(
          process_and_query_lcel,
          inputs=[document_choice, uploaded_files, question_input, session_id],
          outputs=output_text
      )

      clear_user_data_btn.click(delete_user_data, outputs=output_text)

      # Set session_id value on load using gr.Request().session_hash
      demo_lcel.load(lambda request=None: request.session_hash if request and request.session_hash else str(uuid.uuid4()), outputs=session_id)

  demo_lcel.launch(debug=True)
  print("New `demo_lcel` Gradio application created.")

## Gradio Application and the actual pipeline

In [12]:
import gradio as gr
import os
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb # Added this import for explicit client

DEFAULT_DOCS_PATH = './raw_documents'
DEFAULT_DB_PATH = './dbs'

# Initialize the embedding model once globally
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

def rag_pipeline(document_choice: str, uploaded_files, question: str, session_id: str):
    global embedding_model # Use the globally initialized embedding model

    current_vector_db = None

    # Define session-specific paths
    SESSION_USER_DOCS_PATH = os.path.join('./user_docs', session_id)
    SESSION_USER_DBS_PATH = os.path.join('./user_dbs', session_id)

    # Create directories if they don't exist for the current session
    os.makedirs(SESSION_USER_DOCS_PATH, exist_ok=True)
    os.makedirs(SESSION_USER_DBS_PATH, exist_ok=True)

    if document_choice == 'default':
        print(f"[{session_id}] Using default documents and vector DB.")
        if not os.path.exists(DEFAULT_DB_PATH):
          print("Default vector database not found. Creating....")
          load_chunk_add_to_vector(r'./raw_documents', r'./dbs')
          os.system("ls -al /dbs")

        # Explicitly create and pass a PersistentClient for the default DB
        default_client = chromadb.PersistentClient(path=DEFAULT_DB_PATH)
        current_vector_db = Chroma(client=default_client, # Pass the explicit client
                                   persist_directory=DEFAULT_DB_PATH,
                                   embedding_function=embedding_model)

    elif document_choice == 'upload':
        print(f"[{session_id}] Processing uploaded documents.")
        if not uploaded_files:
            return "Please upload PDF files for processing."
        print(f"{SESSION_USER_DOCS_PATH}\n{SESSION_USER_DBS_PATH}")
        # Clear previous user documents for this session
        print(f"For session {session_id} chroma db path: {SESSION_USER_DOCS_PATH}")
        if os.path.exists(SESSION_USER_DOCS_PATH):
            shutil.rmtree(SESSION_USER_DOCS_PATH)
        os.makedirs(SESSION_USER_DOCS_PATH, exist_ok=True)

        # Save uploaded files to the session's document folder
        saved_file_paths = []
        for uploaded_file in uploaded_files:
            file_name = os.path.basename(uploaded_file.name)
            save_path = os.path.join(SESSION_USER_DOCS_PATH, file_name)
            shutil.copyfile(uploaded_file.name, save_path)
            saved_file_paths.append(save_path)
            print(f"[{session_id}] Saved {file_name} to {SESSION_USER_DOCS_PATH}")

        # Load, chunk, and add to the session's vector DB
        user_documents = load_documents(SESSION_USER_DOCS_PATH)
        user_chunks = chunkizer(documents=user_documents, chunk_size=1000, chunk_overlap=200)
        current_vector_db = add_to_vector_db(SESSION_USER_DBS_PATH,
                                             chunks=user_chunks,
                                             embedding_model=embedding_model,
                                             purge=False)

    else:
        return f"Invalid document choice. Please select 'default' or 'upload'. Chosed value {document_choice}"

    if not current_vector_db:
        return "Error: Could not initialize vector database."

    # Perform retrieval
    print(f"[{session_id}] Performing similarity search for: {question}")
    retrieved_docs = vector_db_similarity_search(current_vector_db, query=question, k=5)

    # Combine retrieved documents into a context string
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Generate response using LLM
    system_role = "You are a helpful assistant. Use the following context to answer the user's question. If the answer is not in the context, say 'I cannot find the answer in the provided documents.'"

    full_query = f"Context: {context}\n\nQuestion: {question}"
    print(f"[{session_id}] Generating response for query: {full_query}")
    response = generate_response(system_role, full_query)

    return response

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
choice_map = {
            'The hackathon intent (AST-1, AST-2)': 'default',
            'User defined knowledge base':'upload'
      }
choice_map.keys()
#get the key at the nth location
list(choice_map.keys())[1]

In [ ]:
if True:#def without_lecl():
  with gr.Blocks() as demo:
      gr.Markdown("# RAG Document Query Interface")

      #session_id = gr.State(value="") # To store the session ID
      import uuid
      session_id = gr.State(value=str(uuid.uuid4())) # To store the session ID
      choice_map = {
            'The hackathon intent (AST-1, AST-2)': 'default',
            'User defined knowledge base':'upload'
      }

      def translate_choice(choice:str):
        return choice_map[choice]

      with gr.Row():
          document_choice = gr.Radio(
              choice_map.keys(),
              label="Choose Document Source",
              value=list(choice_map.keys())[0]
          )

      with gr.Row() as upload_section:
          uploaded_files = gr.File(
              label="Upload PDF Documents (for 'upload' option only)",
              file_count="multiple",
              file_types=[".pdf"],
              visible=False # Initially hidden
          )
          upload_btn = gr.Button("Process Uploaded Documents", visible=False)

      def toggle_upload_visibility(choice):
          choice = translate_choice(choice)
          if choice == 'upload':
              return gr.update(visible=True), gr.update(visible=True)
          else:
              return gr.update(visible=False), gr.update(visible=False)

      document_choice.change(
          toggle_upload_visibility,
          inputs=document_choice,
          outputs=[uploaded_files, upload_btn]
      )

      with gr.Row() as delete_other_user_data:
          clear_user_data_btn = gr.Button("Clear all user data", visible=True)

      def delete_user_data():
        folder_paths = ['./user_docs','./user_dbs']
        deleted_folders = []
        for folder in folder_paths:
            if os.path.exists(folder):
                try:
                    shutil.rmtree(folder)
                    deleted_folders.append(folder)
                    os.makedirs(folder, exist_ok=True) # Recreate empty directory
                except Exception as e:
                    print(f"Error deleting folder {folder}: {e}")
        if deleted_folders:
            return f"Deleted data in: {', '.join(deleted_folders)}"
        else:
            return "No user data folders found to clear."

      with gr.Row():
          question_input = gr.Textbox(
              label="Find me in the documents:",
              placeholder="Enter your question here..."
          )
      with gr.Row():
          submit_btn = gr.Button("Get Answer")
      with gr.Row():
          output_text = gr.Textbox(label="Response", interactive=False, lines=10)

      def process_and_query(doc_choice, files, query_text, current_session_id):
          print(document_choice)
          doc_choice = translate_choice(doc_choice)
          print(document_choice)
          output_text.value = "" # Clear the output text
          if doc_choice == 'upload' and not files:
              return "Please upload documents or select 'default' source."
          # Pass the session_id to rag_pipeline
          print(f"Starting the RAG for user specified documents...")
          return rag_pipeline(doc_choice, files, query_text, current_session_id)

      submit_btn.click(
          process_and_query,
          inputs=[document_choice, uploaded_files, question_input, session_id],
          outputs=output_text
      )

      # Trigger processing for uploaded files if the upload button is clicked
      upload_btn.click(
          process_and_query,
          inputs=[document_choice, uploaded_files, question_input, session_id],
          outputs=output_text
      )

      clear_user_data_btn.click(delete_user_data, outputs=output_text)

      # Set session_id value on load using gr.Request().session_hash
      demo.load(lambda request=None: request.session_hash if request and request.session_hash else str(uuid.uuid4()), outputs=session_id)

  demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://581de159e7ea2d0d69.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Starting the RAG for user specified documents...
[5830e7c0-98fd-4aad-a6f8-30917fbf99cc] Using default documents and vector DB.
[5830e7c0-98fd-4aad-a6f8-30917fbf99cc] Performing similarity search for: what is the importance of testing in software development
Similiarity search returned 5 chunks
[5830e7c0-98fd-4aad-a6f8-30917fbf99cc] Generating response for query: Context: ensure that the software functions as intended, meets user expectations, and operates 
reliably in various scenarios. 
Testing provides several benefits to software development: 
● (i) Error Detection: Testing helps identify bugs, errors, and unexpected behavior 
in the software  
● (ii) Verification and Validation : Testing validates that the software meets the 
specified requirements and performs as expected. Verification involves checking 
if the software conforms to the design and development specifications, while 
validation ensures that it meets the user's needs  
● (iii) Risk Mitigation: Testing helps mitigate r

In [ ]:
options = ['with_lecl', 'without_lecl']
choice = options[1]

if choice == 'with_lecl':
  pass#with_lecl()
elif choice == 'without_lecl':
  without_lecl()
else:
  print(f"Invalid choice. Choices are {options}")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5f6e788dfe52ad6e2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Starting the RAG for user specified documents...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5f6e788dfe52ad6e2a.gradio.live
